# scikit-learn: learning patterns and testing predictions

A labeled dataset suggests that future outcomes may be predictable. Establish what a trivial model can do, learn preprocessing only from training data, compare candidates within training folds and evaluate the chosen workflow on untouched examples. Follow the same separation of roles through classification and regression.

A chart helps us inspect patterns. A machine-learning model uses examples to predict something about a new observation. The essential question is whether those predictions work on data the model did not learn from.

We will classify iris flowers using a small real dataset packaged with scikit-learn, compare against a baseline, tune a pipeline using training data, and evaluate it on a held-out test set. Later sections introduce regression, mixed-column preprocessing, clustering, and dimensionality reduction.

Every example runs offline after installation. Run cells from top to bottom. These compact examples teach the workflow; a small benchmark score is not evidence that a model will work in a new setting.

## 1. Install and import

If needed, run `%pip install scikit-learn pandas numpy matplotlib` in a separate cell. The package is installed as `scikit-learn` but imported as `sklearn`.

| Method | Purpose |
| --- | --- |
| `fit(X, y)` | Learn from features and known targets |
| `predict(X)` | Predict targets for feature rows |
| `transform(X)` | Convert features using a learned transformation |
| `fit_transform(X)` | Learn a transformation and apply it to its training data |

`X` is usually a two-dimensional table: rows are examples and columns are features. `y` holds one target per row.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("scikit-learn:", sklearn.__version__)

## 2. Load features and targets

The Iris dataset contains flower measurements and three species labels. The numeric target codes correspond to names in `target_names`; they are category identifiers, not quantities.

`as_frame=True` preserves feature names in a Pandas DataFrame. The target must stay out of the feature table: including the answer would make evaluation meaningless.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
class_names = iris.target_names
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", dict(enumerate(class_names)))
print(X.head())

## 3. Split before learning from the data

Reserve 20% of the rows for the final test. Stratification keeps class proportions similar across the two subsets. A fixed seed makes the split reproducible.

We use only training rows for exploration, preprocessing, and model selection. A random split is appropriate for this teaching example; time-ordered data or repeated measurements from the same person need a split that respects that structure.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))
print("Training class counts:", y_train.value_counts().sort_index().to_dict())
print("Training missing values:", X_train.isna().sum().to_dict())
print(X_train.describe().round(2))

## 4. Inspect training data

Plot two measurements, coloring by the known training labels. Visual separation suggests useful features, but this plot cannot replace an evaluation on unseen rows.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
for label, name in enumerate(class_names):
    rows = y_train == label
    ax.scatter(X_train.loc[rows, "petal length (cm)"], X_train.loc[rows, "petal width (cm)"],
               label=name, alpha=0.7)
ax.set(xlabel="Petal length (cm)", ylabel="Petal width (cm)", title="Training flowers only")
ax.legend()
plt.show()

## 5. Find out whether a sophisticated model is needed

Predict what the baseline uses and ignores. Compare its score with class balance before interpreting a learned model’s improvement. Accuracy alone may reward a model that ignores a small but important class.

A model needs a useful comparison. `DummyClassifier(strategy="most_frequent")` always chooses a most frequent training class.

Five-fold stratified cross-validation repeatedly fits on four training folds and scores on the remaining fold. The final test set stays untouched. Fold-to-fold variation describes these splits; it is not automatically a confidence interval.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline = DummyClassifier(strategy="most_frequent")
baseline_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="accuracy")
print("Baseline fold accuracies:", baseline_scores)
print(f"Baseline mean accuracy: {baseline_scores.mean():.3f}")

## 6. Keep preprocessing inside the learning boundary

Identify which quantities the scaler learns. Explain why fitting it before a split would use information from held-out rows. A pipeline keeps the transformation paired with the estimator during fitting and later prediction.

Standard scaling subtracts each feature's training mean and divides by its training standard deviation. This makes differently scaled features more comparable for regularized logistic regression. Despite its name, logistic regression is a classifier.

A Pipeline fits the scaler inside each cross-validation training fold, then applies that fitted scaler to the validation fold. Scaling the entire dataset before splitting would leak information about held-out rows.

In [ ]:
classifier = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
model_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="accuracy")
cv_summary = pd.DataFrame({
    "model": ["Most frequent baseline", "Scaled logistic regression"],
    "mean_cv_accuracy": [baseline_scores.mean(), model_scores.mean()],
    "fold_std": [baseline_scores.std(), model_scores.std()],
})
print(cv_summary.round(3))

## 7. Choose settings without consulting the final test set

Predict how many train/validation fits the search performs from its candidates and folds. Inspect score variation as well as the best mean. Folds must reflect the sampling structure; time or repeated entities can invalidate ordinary random folds.

`C` controls inverse regularization strength: smaller values impose stronger regularization. `model__C` addresses the parameter of the step named `model`.

Grid search compares three candidates using the same training folds. It then refits the chosen pipeline on all training rows. Its best cross-validation score helped select the candidate and can be optimistic; the reserved test set provides the final check.

In [ ]:
search = GridSearchCV(classifier, param_grid={"model__C": [0.1, 1.0, 10.0]},
                      scoring="accuracy", cv=cv, n_jobs=1, refit=True)
search.fit(X_train, y_train)
print("Chosen parameters:", search.best_params_)
print(f"Best training CV accuracy: {search.best_score_:.3f}")
print(pd.DataFrame(search.cv_results_)[["param_model__C", "mean_test_score", "std_test_score"]])
best_model = search.best_estimator_

## 8. Evaluate the chosen model once on the test set

Accuracy is the fraction of correct predictions. Precision asks how often predictions of a class are correct; recall asks how many actual members of that class were recovered. F1 combines precision and recall. Support is the number of test examples in the class.

Macro averages weight classes equally. Weighted averages weight them by support. This test set has only 30 flowers, so one error changes accuracy by about 3.3 percentage points. Do not repeatedly tune against its results.

In [ ]:
test_predictions = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Held-out accuracy: {test_accuracy:.3f}")
print(classification_report(y_test, test_predictions, labels=[0, 1, 2],
                            target_names=class_names, zero_division=0))

## 9. Translate aggregate accuracy into kinds of mistakes

Read the confusion-matrix axis labels before choosing a cell to interpret. Explain one false positive and one false negative in the dataset’s class terms. Decide which mistake would matter more in a proposed use case before choosing a threshold.

Rows are true classes and columns are predicted classes. The diagonal contains correct predictions; off-diagonal entries show which classes were confused. This explains mistakes that a single accuracy value hides.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=[0, 1, 2],
                                       display_labels=class_names, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Held-out iris predictions")
plt.show()

## 10. Predict for new feature rows

New rows must have the same feature names, order, meaning, and units as the training data. These two measurements are invented examples. Passing them through the pipeline automatically applies the fitted scaler.

`predict_proba` returns model-estimated class probabilities in `classes_` order. High probability does not guarantee correctness, especially for observations unlike the training data.

In [ ]:
new_flowers = pd.DataFrame([[5.1, 3.5, 1.4, 0.2], [6.5, 3.0, 5.2, 2.0]], columns=X.columns)
new_predictions = best_model.predict(new_flowers)
probabilities = best_model.predict_proba(new_flowers)
print("Predicted species:", class_names[new_predictions])
print(pd.DataFrame(probabilities, columns=class_names[best_model.classes_]).round(3))

## 11. Regression predicts a numerical value

Classification predicts a category; regression predicts a number. Here we use generated data with a linear signal and noise so the example is self-contained. Target values have arbitrary units.

Compare Ridge regression to a training-mean baseline. Mean absolute error (MAE) is the average absolute prediction error, in target units; smaller is better. R² compares squared error against a constant equal to the evaluation targets' mean. It can be negative.

In [ ]:
X_reg, y_reg = make_regression(n_samples=250, n_features=5, n_informative=3,
                                noise=20, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
regressor = Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))])
reg_baseline = DummyRegressor(strategy="mean")
regression_rows = []
for name, estimator in [("Training mean", reg_baseline), ("Ridge", regressor)]:
    estimator.fit(Xr_train, yr_train)
    predictions = estimator.predict(Xr_test)
    regression_rows.append({"model": name, "MAE": mean_absolute_error(yr_test, predictions),
                            "R2": r2_score(yr_test, predictions)})
print(pd.DataFrame(regression_rows).round(3))
reg_predictions = regressor.predict(Xr_test)

## 12. Inspect regression errors

A residual here is actual minus predicted. A good summary score can hide systematic errors; the residual plot helps inspect whether errors change with prediction size. A small test sample cannot establish that every model assumption holds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].scatter(yr_test, reg_predictions, alpha=0.7)
limits = [min(yr_test.min(), reg_predictions.min()), max(yr_test.max(), reg_predictions.max())]
axes[0].plot(limits, limits, color="black", linestyle="--")
axes[0].set(xlabel="Actual target", ylabel="Predicted target", title="Predictions vs actual values")
axes[1].scatter(reg_predictions, yr_test - reg_predictions, alpha=0.7)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="Predicted target", ylabel="Residual: actual − predicted", title="Inspect errors")
plt.show()

## 13. Different columns need different preprocessing

Real tables often mix numerical and categorical features. `ColumnTransformer` applies separate transformations to selected columns, and Pipeline connects the transformed features to an estimator.

This tiny invented table demonstrates the mechanics only. Median imputation and scaling handle the numerical column; one-hot encoding represents category membership. `handle_unknown="ignore"` allows a new category, encoding it as zeros across the known category columns. That behavior prevents an exception but does not teach the model what the new category means.

In [ ]:
mixed_train = pd.DataFrame({
    "area": [40.0, 55.0, np.nan, 80.0, 95.0, 65.0],
    "district": ["North", "South", "North", "South", "North", "South"],
})
mixed_target = np.array([100, 125, 120, 170, 205, 145])
numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
preprocess = ColumnTransformer([
    ("numeric", numeric_steps, ["area"]),
    ("category", OneHotEncoder(handle_unknown="ignore"), ["district"]),
])
mixed_model = Pipeline([("preprocess", preprocess), ("model", Ridge(alpha=1.0))])
mixed_model.fit(mixed_train, mixed_target)
mixed_new = pd.DataFrame({"area": [70.0, np.nan], "district": ["North", "West"]})
print("Illustrative predictions:", mixed_model.predict(mixed_new).round(2))
print("Transformed features:", mixed_model.named_steps["preprocess"].get_feature_names_out())

## 14. Clustering finds groups without target labels

K-means groups observations by distance to cluster centers. Scaling matters because distance depends on feature units. Here we fit only the Iris training measurements, without passing species labels.

We choose three clusters for illustration; that choice is not a discovery. Cluster IDs are arbitrary, and cluster 0 does not mean species 0. Clustering performance is not measured by directly comparing those numeric IDs to class codes.

In [ ]:
cluster_model = Pipeline([
    ("scale", StandardScaler()),
    ("cluster", KMeans(n_clusters=3, n_init=10, random_state=42)),
])
cluster_ids = cluster_model.fit_predict(X_train)
print("Cluster sizes:", np.bincount(cluster_ids))

## 15. PCA projects many features into fewer dimensions

Principal component analysis creates directions that capture variation. We standardize the four Iris measurements, then project them into two dimensions. The explained variance ratios report how much standardized variation each component captures.

The plot colors points by the clusters found above. A two-dimensional projection can hide differences in discarded dimensions; it is not a classification evaluation.

In [ ]:
projection = Pipeline([("scale", StandardScaler()), ("pca", PCA(n_components=2))])
coordinates = projection.fit_transform(X_train)
print("Explained variance ratios:", projection.named_steps["pca"].explained_variance_ratio_.round(3))
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
for cluster_id in np.unique(cluster_ids):
    rows = cluster_ids == cluster_id
    ax.scatter(coordinates[rows, 0], coordinates[rows, 1], label=f"Cluster {cluster_id}", alpha=0.7)
ax.set(title="Training flowers: PCA view of K-means groups", xlabel="Principal component 1", ylabel="Principal component 2")
ax.legend()
plt.show()

## 16. Export an evaluation report

Keep the training cross-validation comparison distinct from the final test result. The first panel shows training-fold scores for the original baseline and classifier; the second shows final test errors for the tuned model.

This cell writes a figure and CSV predictions into `outputs`, replacing files with these names if present. Original row IDs make predictions traceable to the bundled dataset.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
report, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
axes[0].bar(["Baseline", "Logistic regression"], cv_summary["mean_cv_accuracy"], color=["gray", "tab:blue"])
axes[0].set(ylim=(0, 1), ylabel="Mean validation accuracy", title="Training cross-validation, before tuning")
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=[0, 1, 2],
    display_labels=class_names, cmap="Blues", colorbar=False, ax=axes[1])
axes[1].set_title(f"Tuned model: test accuracy {test_accuracy:.3f}")
report.suptitle("Iris classification evaluation")
for extension in ["png", "svg"]:
    destination = output_dir / f"sklearn_iris_report.{extension}"
    report.savefig(destination, dpi=150, bbox_inches="tight")
    print("Saved:", destination)
prediction_table = pd.DataFrame({"row_id": X_test.index, "actual": class_names[y_test.to_numpy()],
                                  "predicted": class_names[test_predictions]})
prediction_table.to_csv(output_dir / "sklearn_iris_predictions.csv", index=False)
print("Saved:", output_dir / "sklearn_iris_predictions.csv")
plt.show()

## 17. Practice and common mistakes

1. Use a different classifier inside a Pipeline and compare cross-validation scores on training data. Plan the comparison before consulting a final test set.
2. Change regression noise from 20 to 40 and explain why prediction errors increase.
3. Inspect `best_model.named_steps["model"].coef_`. Remember that these coefficients apply to standardized features.
4. Try two and four clusters. Explain why choosing a cluster count requires more than an attractive plot.

| Observation | What to check |
| --- | --- |
| Suspiciously excellent validation scores | Target leakage, duplicate rows, group overlap, and preprocessing outside folds |
| Error about a one-dimensional input | Predict with a table of rows, even for one example |
| Feature-name mismatch | Preserve the training feature schema and units |
| Good training score, poor validation score | Overfitting or differences between training and evaluation data |
| Accuracy hides poor minority-class performance | Inspect per-class precision, recall, support, and the confusion matrix |
| New categories cause an encoding error | Decide how unknown categories should be handled |
| Results vary between runs | Set random seeds for randomized splits and estimators |

## Further reading

The official [getting started guide](https://scikit-learn.org/stable/getting_started.html) introduces the estimator interface. See [common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html) for data leakage, [pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html) for preprocessing, and [bundled datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html) for dataset descriptions and provenance.

## Explain what the evidence supports

Compare the selected model with its baseline, identify one important error type and explain why the evaluation examples were not used for preprocessing or tuning.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: compare pipelines on a messy time-ordered export

A subscription-scoring export has repeated rows, malformed and missing usage, missing labels, a new category appearing late, and a tempting field populated only after the outcome. Build a time-ordered development/test workflow on 1,000 generated records with a mild late-period shift. This is a synthetic operational scenario, not a real churn dataset; it is designed to expose boundary mistakes while keeping training small.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a feature that predicts too well

`refund_after_outcome` is generated from the label. It would look highly useful but is unavailable when a churn prediction is made. Exclude it by an explicit feature allowlist before preprocessing or tuning. Also exclude row IDs, which index time but are not intended predictors here. Missing labels cannot be imputed as though they were ordinary features.

Fit imputation, scaling and categorical vocabulary only on the training period. An unseen late category should be handled deliberately rather than silently refitting the encoder on the test set.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(63);p2_n=1000
p2_usage=p2_rng.gamma(2,3,p2_n);p2_tenure=p2_rng.integers(1,36,p2_n)
p2_region=p2_rng.choice(['East','West','North'],p2_n).astype(object);p2_region[900:]='NewRegion'
p2_logit=1.8-.28*p2_usage-.035*p2_tenure+.5*(p2_region=='West')+.3*(np.arange(p2_n)>=800)
p2_target=p2_rng.binomial(1,1/(1+np.exp(-p2_logit)))
p2_raw=pd.DataFrame({'row_id':np.arange(p2_n),'usage':p2_usage.astype(object),'tenure':p2_tenure,
 'region':p2_region,'churn':p2_target.astype(float),'refund_after_outcome':p2_target+np.random.default_rng(64).normal(0,.01,p2_n)})
p2_raw.loc[10:19,'usage']='malformed';p2_raw.loc[20:39,'usage']=None;p2_raw.loc[3:7,'churn']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().copy();assert p2_clean.row_id.is_unique
p2_clean['usage']=pd.to_numeric(p2_clean.usage,errors='coerce')
p2_missing_label=int(p2_clean.churn.isna().sum());p2_clean=p2_clean.dropna(subset=['churn'])
p2_clean['churn']=p2_clean.churn.astype(int)
p2_train=p2_clean[p2_clean.row_id<600];p2_valid=p2_clean[(p2_clean.row_id>=600)&(p2_clean.row_id<800)];p2_test=p2_clean[p2_clean.row_id>=800]
p2_features=['usage','tenure','region'];assert 'refund_after_outcome' not in p2_features
assert len(p2_train)==595 and len(p2_valid)==len(p2_test)==200
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss,roc_auc_score,average_precision_score

def p2_preprocess():
    return ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),['usage','tenure']),
      ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['region'])],sparse_threshold=0)
def p2_pipeline(estimator):return Pipeline([('prepare',p2_preprocess()),('model',estimator)])

from sklearn.ensemble import RandomForestClassifier
p2_candidates={'linear':p2_pipeline(LogisticRegression(max_iter=1000,random_state=4)),
 'forest':p2_pipeline(RandomForestClassifier(n_estimators=60,max_depth=6,min_samples_leaf=8,random_state=4,n_jobs=1))}
p2_validation=[]
for p2_name,p2_candidate in p2_candidates.items():
    p2_t=time.perf_counter();p2_candidate.fit(p2_train[p2_features],p2_train.churn)
    p2_prob=p2_candidate.predict_proba(p2_valid[p2_features])[:,1]
    p2_validation.append({'candidate':p2_name,'validation_log_loss':log_loss(p2_valid.churn,p2_prob),'fit_seconds':time.perf_counter()-p2_t})
p2_selection=pd.DataFrame(p2_validation).sort_values(['validation_log_loss','candidate'])
p2_name=p2_selection.iloc[0].candidate;p2_selected=p2_candidates[p2_name]
display(p2_selection)

### Work through selection and a bounded prediction batch

Choose the candidate by validation log loss only, then compare it with a prior baseline on the untouched later period. Log loss evaluates probabilities; ROC AUC evaluates ordering, so neither replaces the other. The model's behavior under `NewRegion` is part of this fixed test, not a reason to refit the encoder afterward.

Compare one large `predict_proba` call with 32-row chunks, preserving index order. Chunking bounds the size of intermediate feature matrices but adds call overhead; it is not automatically faster.

In [ ]:
p2_prior=DummyClassifier(strategy='prior').fit(p2_train[p2_features],p2_train.churn)
p2_t=time.perf_counter();p2_full=p2_selected.predict_proba(p2_test[p2_features])[:,1];p2_full_time=time.perf_counter()-p2_t
p2_t=time.perf_counter();p2_batched=np.concatenate([p2_selected.predict_proba(p2_test[p2_features].iloc[i:i+32])[:,1] for i in range(0,len(p2_test),32)]);p2_batch_time=time.perf_counter()-p2_t
assert np.allclose(p2_full,p2_batched)
p2_report=pd.DataFrame([{'model':p2_name,'test_log_loss':log_loss(p2_test.churn,p2_full),'test_auc':roc_auc_score(p2_test.churn,p2_full)},
 {'model':'prior','test_log_loss':log_loss(p2_test.churn,p2_prior.predict_proba(p2_test[p2_features])[:,1]),'test_auc':.5}])
display(p2_report);print({'whole_seconds':p2_full_time,'chunk32_seconds':p2_batch_time,'labels_quarantined':p2_missing_label})
p2_report.to_csv(p2_out/'sklearn_temporal_report.csv',index=False)

### Explain the comparison

The selected model is the winner on one development period, not a universally best churn model. The test includes a mild shift and an unseen category; poor performance may reflect generalization rather than broken preprocessing. The 60-tree cap fixes an interpretable compute budget for the nonlinear alternative.

**Try:** add the leaked field to a separate diagnostic experiment without using its scores for model selection. **Worked reasoning:** an impressive score would establish only that the future-derived field encodes the target. Feature availability at decision time matters more than predictive strength. Keep the final pipeline's allowlist unchanged unless new deployment evidence justifies a field.